# 7.18 Fine-Tuning, Evaluation, and Neural-Network Failure Modes

[Back to neural networks index](../7_neural_networks.ipynb)

This final notebook closes the neural-networks block by focusing on adaptation, evaluation, and the ways neural models can fail even when they seem to perform well at first glance.


## 7.18.1 Freezing versus full fine-tuning

### Why It Matters
Fine-tuning decisions change both compute cost and which parts of the model can adapt to the new task.


In [ ]:
import torch
from torchvision.models import resnet18

frozen_model = resnet18(weights=None)
for param in frozen_model.parameters():
    param.requires_grad = False
frozen_model.fc = torch.nn.Linear(frozen_model.fc.in_features, 5)

full_model = resnet18(weights=None)
full_model.fc = torch.nn.Linear(full_model.fc.in_features, 5)

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

{"frozen_trainable_params": count_trainable(frozen_model), "full_trainable_params": count_trainable(full_model)}


### Interpretation
Freezing most of the model makes adaptation cheaper, but it also limits how much the representation can change.


## 7.18.2 Task evaluation by modality

### Why It Matters
Evaluation metrics depend on the task: classification accuracy, regression loss, perplexity-like language objectives, or image-specific error analysis.


In [ ]:
from sklearn.metrics import accuracy_score, mean_squared_error

classification_truth = [0, 1, 1, 0]
classification_pred = [0, 1, 0, 0]
regression_truth = [1.2, 2.0, 3.1]
regression_pred = [1.0, 2.4, 2.8]
{"classification_accuracy": round(float(accuracy_score(classification_truth, classification_pred)), 3), "regression_mse": round(float(mean_squared_error(regression_truth, regression_pred)), 3)}


### Interpretation
There is no single universal evaluation number for all neural-network tasks.


## 7.18.3 Distribution shift

### Why It Matters
A model trained on one input distribution can degrade on a shifted one. Here the same classifier is tested on an easier and then a shifted version of the data.


In [ ]:
import torch
from torch import nn

torch.manual_seed(41)
X_train = torch.randn(160, 3)
y_train = ((X_train[:, 0] + X_train[:, 1]) > 0).float().unsqueeze(1)
X_shift = torch.randn(160, 3) + torch.tensor([1.2, -1.0, 0.0])
y_shift = ((X_shift[:, 0] + X_shift[:, 1]) > 0).float().unsqueeze(1)
model = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
opt = torch.optim.Adam(model.parameters(), lr=0.03)
loss_fn = nn.BCELoss()
for _ in range(180):
    opt.zero_grad()
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    opt.step()
with torch.no_grad():
    train_acc = (((model(X_train) > 0.5).float() == y_train).float().mean()).item()
    shift_acc = (((model(X_shift) > 0.5).float() == y_shift).float().mean()).item()
{"train_accuracy": round(float(train_acc), 3), "shifted_accuracy": round(float(shift_acc), 3)}


### Interpretation
Strong training performance does not guarantee stability when the data-generating process changes.


## 7.18.4 Bias and hallucination-style failures

### Why It Matters
Neural models can output confident answers even when their evidence is weak or absent. A toy probability example makes that calibration risk visible.


In [ ]:
import pandas as pd
import torch

logits = torch.tensor([[4.2, -1.0], [0.3, 0.1], [2.5, 2.4], [3.0, -2.0]])
truth = torch.tensor([0, 1, 1, 1])
probs = torch.softmax(logits, dim=1)
pred = probs.argmax(dim=1)
confidence = probs.max(dim=1).values

pd.DataFrame({
    "truth": truth.tolist(),
    "prediction": pred.tolist(),
    "confidence": confidence.round(decimals=3).tolist(),
    "correct": (pred == truth).tolist(),
})


### Interpretation
A confident distribution is not the same thing as a correct answer. That distinction matters even more in generative models.


## 7.18.5 Interpretability and monitoring basics

### Why It Matters
You usually cannot inspect every parameter directly, so monitoring summaries and simple attribution tools become part of the workflow.


In [ ]:
import torch
from torch import nn

model = nn.Sequential(nn.Linear(4, 6), nn.ReLU(), nn.Linear(6, 1), nn.Sigmoid())
x = torch.tensor([[0.5, -1.0, 0.2, 1.3]], requires_grad=True)
pred = model(x)
pred.backward()
{"prediction": round(float(pred.item()), 4), "input_gradient": x.grad.flatten().round(decimals=4).tolist()}


### Interpretation
Input gradients are not a full explanation method, but they make the idea of feature sensitivity concrete.


## 7.18.6 Capstone comparison across neural architectures

### Why It Matters
The course closes by comparing parameter counts and output shapes across a feedforward net, a CNN, a recurrent model, and a transformer-style classifier.


In [ ]:
import torch
from torch import nn

mlp = nn.Sequential(nn.Linear(10, 16), nn.ReLU(), nn.Linear(16, 2))
cnn = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(4 * 8 * 8, 2))
rnn = nn.GRU(6, 8, batch_first=True)
transformer = nn.TransformerEncoder(nn.TransformerEncoderLayer(d_model=12, nhead=3, batch_first=True), num_layers=1)

summary = {
    "mlp_params": sum(p.numel() for p in mlp.parameters()),
    "cnn_params": sum(p.numel() for p in cnn.parameters()),
    "gru_params": sum(p.numel() for p in rnn.parameters()),
    "transformer_params": sum(p.numel() for p in transformer.parameters()),
}
summary


### Interpretation
The architectures differ because the data structure differs. The course is really a study of matching inductive bias to problem shape.
